## Below is a claims pipeline that relies upon google/bigbird-roberta-base.

It handles long claim narratives using BigBird’s sparse attention mechanism, which allows efficient processing of sequences up to 4096 tokens — far beyond the 256‑token limit of Bio_ClinicalBERT, and typically faster than Clinical‑Longformer on similar hardware.

## 00 - imports and device

In [1]:
import sys
print(sys.executable)
print(sys.version)

C:\Users\marke\projects\gpu-transformers\.venv\Scripts\python.exe
3.11.9 (tags/v3.11.9:de54cf5, Apr  2 2024, 10:12:12) [MSC v.1938 64 bit (AMD64)]


In [2]:
import torch
import pandas as pd
import numpy as np
from datasets import Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback,
    LongformerForSequenceClassification
)

import shap
import matplotlib.pyplot as plt
import seaborn as sns

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DEVICE

'cuda'

In [3]:
# windows or wsl
import platform
print(platform.system())
import torch
print(torch.__version__)
print(torch.version.cuda)
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

Windows
2.12.0.dev20260408+cu128
12.8
True
NVIDIA GeForce RTX 5060 Laptop GPU


In [4]:
%matplotlib inline

## 01 - Load your dataset

In [5]:
df = pd.read_json("synthetic_claims.json")
df.head()

,id,severity,theme,department,claim_text
0,1,high,diagnostic_error,ED/Cardiology,"The patient, a 68-year-old male with hypertens..."
1,2,high,failure_to_escalate,Surgery/ICU,A 54-year-old female underwent elective laparo...
2,3,low,diagnostic_error,ED/Fracture Clinic,A 32-year-old male attended A&E after falling ...
3,4,high,delay_in_treatment,ED/Respiratory,A 76-year-old patient with COPD presented with...
4,5,high,fetal_monitoring_failure,Maternity,A 29-year-old primigravida presented in labour...


## 02 - Map severity

In [6]:
severity_map = {"low": 0, "moderate": 1, "high": 2}
df["severity_label"] = df["severity"].map(severity_map)

## 03 - train/validation split

In [7]:
train_df, val_df = train_test_split(df, test_size=0.2, random_state=42)

In [8]:
train_df, val_df

(    id  severity                  theme   department  \
 55  56  moderate  communication_failure           GP   
 88  89      high         surgical_error      Theatre   
 26  27      high       diagnostic_error           ED   
 42  43      high    failure_to_escalate           ED   
 69  70       low   administrative_delay  Outpatients   
 ..  ..       ...                    ...          ...   
 60  61      high       diagnostic_error           ED   
 71  72  moderate       medication_error     Pharmacy   
 14  15      high       diagnostic_error           ED   
 92  93      high    failure_to_escalate           ED   
 51  52  moderate       medication_error     Pharmacy   
 
                                            claim_text  severity_label  
 55  A patient was not informed of abnormal liver f...               1  
 88  A surgical instrument was retained during proc...               2  
 26  A 39-year-old female presented with chest pain...               2  
 42  A patient with se

## 04 - Convert to huggingface dataset

In [9]:
train_ds = Dataset.from_pandas(train_df)
val_ds = Dataset.from_pandas(val_df)

## 05 Load tokenisers and models

In [10]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

MODEL_NAME = "google/bigbird-roberta-base"

long_tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=3
).to(DEVICE)

model.gradient_checkpointing_enable()

Some weights of BigBirdForSequenceClassification were not initialized from the model checkpoint at google/bigbird-roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


## 06 - Tokenisation functions

In [11]:
def tokenize(batch):
    return long_tokenizer(
        batch["claim_text"],
        padding="max_length",
        truncation=True,
        max_length=4096 # i.e. the max
    )

## 07 - Apply tokenisation

In [12]:
long_train = train_ds.map(tokenize, batched=True)
long_val = val_ds.map(tokenize, batched=True)

Map:   0%|          | 0/80 [00:00<?, ? examples/s]

Map:   0%|          | 0/20 [00:00<?, ? examples/s]

## 08 - Rename label column

In [13]:
long_train = long_train.rename_column("severity_label", "labels")
long_val = long_val.rename_column("severity_label", "labels")

## 09 - Remove unused columns

In [14]:
cols_to_remove = ["id", "severity", "theme", "department", "claim_text"]

long_train = long_train.remove_columns(cols_to_remove)
long_val = long_val.remove_columns(cols_to_remove)

## 10 - Set format for pytorch

In [15]:
long_train.set_format("torch")
long_val.set_format("torch")

## 11 - Metrics

In [16]:
def compute_metrics(pred):
    labels = pred.label_ids
    preds = np.argmax(pred.predictions, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1_macro": f1_score(labels, preds, average="macro")
    }

## 12 - Training arguments

In [17]:
training_args = TrainingArguments(
    output_dir="./bigbird-claims",
    num_train_epochs=20,
    per_device_train_batch_size=2,     # BigBird is memory-heavy at long seq lengths
    per_device_eval_batch_size=2,
    learning_rate=2e-5,
    eval_strategy="epoch",
    logging_strategy="epoch",
    save_strategy="epoch",
    report_to="none",
    load_best_model_at_end=True,
    metric_for_best_model="loss",
    greater_is_better=False,
    warmup_steps=200,
    max_grad_norm=1.0,
    # Use BF16 on RTX 5060 (better than FP16)
    bf16=True,
    # Essential for long-sequence models
    gradient_checkpointing=True,
    # Faster fused optimizer
    optim="adamw_torch",
    # Faster dataloading
    dataloader_num_workers=4,
    # Avoid saving 20 checkpoints
    save_total_limit=2,
    # Avoid safetensors issues with BigBird
    save_safetensors=False
)


## 13 - Trainers

In [18]:
long_trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=long_train,
    eval_dataset=long_val,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=1)]
)

## 14 - train and evaluation

In [19]:
long_trainer.train()
long_metrics = long_trainer.evaluate()

C:\Users\marke\projects\gpu-transformers\.venv\Lib\site-packages\torch\_dynamo\eval_frame.py:1269: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro
1,1.135200,1.085156,0.400000,0.190476
2,0.995900,0.797461,0.550000,0.236559
3,0.700400,0.443213,0.900000,0.613757
4,0.390400,0.169128,0.950000,0.964519
5,0.229600,0.239162,0.950000,0.964519


C:\Users\marke\projects\gpu-transformers\.venv\Lib\site-packages\torch\_dynamo\eval_frame.py:1269: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
C:\Users\marke\projects\gpu-transformers\.venv\Lib\site-packages\torch\_dynamo\eval_frame.py:1269: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variant

In [20]:
print("=== MODEL PERFORMANCE ===")

print(f"BigBird Accuracy: {long_metrics['eval_accuracy']:.3f}")
print(f"BigBird F1:       {long_metrics['eval_f1_macro']:.3f}")

print("\nInterpretation:")
print("- BigBird processes long claim narratives (up to 4096 tokens) → more stable severity predictions.")


=== MODEL PERFORMANCE ===
BigBird Accuracy: 0.950
BigBird F1:       0.965

Interpretation:
- BigBird processes long claim narratives (up to 4096 tokens) → more stable severity predictions.


In [21]:
# check which one it got wrong
long_preds = long_trainer.predict(long_val)
long_pred_labels = np.argmax(long_preds.predictions, axis=-1)
long_true_labels = long_preds.label_ids

long_misclassified_idx = np.where(long_pred_labels != long_true_labels)[0]
long_misclassified = val_df.iloc[long_misclassified_idx]
long_misclassified

,id,severity,theme,department,claim_text,severity_label
30,31,high,diagnostic_error,ED,A 72-year-old male presented with back pain an...,2


## 15 - shap explainer - bigbird

In [22]:
def bigbird_predict_proba(masked_texts):
    # SHAP passes List[List[str]] → convert back to strings
    texts = [" ".join(tokens) for tokens in masked_texts]

    enc = long_tokenizer(
        texts,
        padding=True,
        truncation=True,
        max_length=512,   # SHAP-friendly
        return_tensors="pt"
    ).to(DEVICE)

    with torch.no_grad():
        outputs = model(**enc)
        probs = torch.softmax(outputs.logits, dim=-1)

    return probs.cpu().numpy()


In [23]:
masker = shap.maskers.Text(long_tokenizer)

long_explainer = shap.Explainer(
    bigbird_predict_proba,
    masker,
    algorithm="auto",   # IMPORTANT
    batch_size=8
)

In [24]:
# 4. Explain a claim
#claim = df["claim_text"].iloc[0]
# check the one it got wrong in the validation set
claim_text = val_df.iloc[0]["claim_text"] 
long_shap_values = long_explainer([claim_text], max_evals=200)

Attention type 'block_sparse' is not possible if sequence_length: 134 <= num global tokens: 2 * config.block_size + min. num sliding tokens: 3 * config.block_size + config.num_random_blocks * config.block_size + additional buffer: config.num_random_blocks * config.block_size = 704 with config.block_size = 64, config.num_random_blocks = 3. Changing attention type to 'original_full'...
PartitionExplainer explainer: 2it [00:10, 10.86s/it]               


In [25]:
# get the predicted class
pred_class = bigbird_predict_proba([claim_text]).argmax()
severity_map = {0: "low", 1: "moderate", 2: "high"}
print("Predicted severity:", severity_map[pred_class])

Predicted severity: moderate


In [26]:
# plot the shap for that class
shap.plots.text(long_shap_values[0, :, pred_class])

In [27]:
# plot the shap for that low class
shap.plots.text(long_shap_values[0, :, 0])

In [30]:
# plot the shap for that high class
shap.plots.text(long_shap_values[0, :, 2])

In [29]:
# 5. Visualise
shap.plots.text(long_shap_values[0])